# Generazione multi metric

In [1]:
from pathlib import Path
import numpy as np

# Configurazione
BRT_DIR = Path(
    "/home/alessio-grisorio/TESI/hj_reachability/results/brt"
)

METRICS = [
    "euclidean",
    "dce",
    "ttc",
    "eggert",
    "btn",
    "stn",
]

PERCENTILE = 95.0

AXES = (
    "x_rel",
    "y_rel",
    "theta_rel",
    "v_H",
    "delta_E",
    "v_E",
)

if not METRICS or len(METRICS) != len(set(METRICS)):
    raise ValueError("METRICS deve essere non vuoto e senza duplicati.")

if not 0.0 < PERCENTILE <= 100.0:
    raise ValueError("PERCENTILE deve essere maggiore di 0 e non superiore a 100.")

metric_paths = {
    metric: BRT_DIR / f"{metric}.npz"
    for metric in METRICS
}

# Griglia di riferimento
with np.load(metric_paths[METRICS[0]], allow_pickle=False) as data:
    grid = {axis: data[axis].copy() for axis in AXES}

grid_shape = tuple(grid[axis].size for axis in AXES)
total_nodes = int(np.prod(grid_shape))

V0_mix = np.full(grid_shape, np.inf, dtype=np.float64)
tie_count = np.zeros(grid_shape, dtype=np.uint32)

normalization_scales = {}

# Primo passaggio:
# normalizzazione, minimo progressivo e numero di vincitori per nodo.
for metric, path in metric_paths.items():
    with np.load(path, allow_pickle=False) as data:
        for axis in AXES:
            if not np.array_equal(data[axis], grid[axis]):
                raise ValueError(
                    f"{metric}: asse {axis} diverso dalla griglia di riferimento."
                )

        values = np.asarray(data["V0"], dtype=np.float64)

    if values.shape != grid_shape:
        raise ValueError(f"{metric}: forma di V0 incompatibile con la griglia.")

    if not np.all(np.isfinite(values)):
        raise ValueError(f"{metric}: V0 contiene NaN o valori infiniti.")

    scale = float(np.percentile(np.abs(values), PERCENTILE))

    if not np.isfinite(scale) or scale <= 0.0:
        raise ValueError(
            f"{metric}: scala di normalizzazione non valida ({scale})."
        )

    normalization_scales[metric] = scale
    values /= scale

    lower = values < V0_mix
    equal = values == V0_mix

    # Un minimo inferiore sostituisce tutti i vincitori precedenti.
    tie_count[lower] = 1

    # Un minimo identico aggiunge una metrica ai vincitori del nodo.
    tie_count[equal] += 1

    np.minimum(V0_mix, values, out=V0_mix)

    del values, lower, equal

# Secondo passaggio:
# attribuzione equa dei nodi con minimo condiviso.
selection_percentages = {}

for metric, path in metric_paths.items():
    with np.load(path, allow_pickle=False) as data:
        values = np.asarray(data["V0"], dtype=np.float64)

    values /= normalization_scales[metric]

    selected = values == V0_mix

    # Esempio: se due metriche condividono il minimo,
    # ciascuna riceve metà del contributo di quel nodo.
    contribution = np.sum(
        1.0 / tie_count[selected],
        dtype=np.float64,
    )

    selection_percentages[metric] = (
        100.0 * contribution / total_nodes
    )

    del values, selected

# Risultati
print(f"Griglia: {grid_shape}")
print(f"Nodi complessivi: {total_nodes:,}")
print(f"Normalizzazione: percentile {PERCENTILE:g} di |V0|\n")

print(f"{'Metrica':<14} {'Scala P95':>14} {'Quota nel mix':>16}")

for metric in METRICS:
    print(
        f"{metric:<14} "
        f"{normalization_scales[metric]:>14.6g} "
        f"{selection_percentages[metric]:>15.2f}%"
    )

shared_percentage = 100.0 * np.count_nonzero(tie_count > 1) / total_nodes
unsafe_percentage = 100.0 * np.count_nonzero(V0_mix <= 0.0) / total_nodes

print(f"\nSomma delle quote: {sum(selection_percentages.values()):.2f}%")
print(f"Nodi con minimo condiviso: {shared_percentage:.2f}%")
print(f"Nodi unsafe nel V0 misto: {unsafe_percentage:.2f}%")
print(f"Intervallo V0_mix: [{V0_mix.min():.4f}, {V0_mix.max():.4f}]")

Griglia: (30, 17, 48, 8, 11, 8)
Nodi complessivi: 17,233,920
Normalizzazione: percentile 95 di |V0|

Metrica             Scala P95    Quota nel mix
euclidean             12.2151           47.33%
dce                   9.74057           11.57%
ttc                         3            2.97%
eggert                    0.8           12.60%
btn                   10.5904           15.26%
stn                   9.82801           10.27%

Somma delle quote: 100.00%
Nodi con minimo condiviso: 2.07%
Nodi unsafe nel V0 misto: 24.10%
Intervallo V0_mix: [-1.0000, 1.0000]


In [2]:
import sys
import json
from time import perf_counter
import os

os.environ["JAX_PLATFORMS"] = "cpu"

PROJECT_ROOT = BRT_DIR.resolve().parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import jax
import jax.numpy as jnp
import hj_reachability as hj

from hj_reachability.systems.relative_vehicle_6d import RelativeVehicle6D

# Impostazioni salvate nella prima metrica
reference_path = metric_paths[METRICS[0]]

with np.load(reference_path, allow_pickle=False) as data:
    reference_metadata = json.loads(data["metadata_json"].item())

    grid_lo = data["grid_lo"].copy()
    grid_hi = data["grid_hi"].copy()
    periodic_dims = tuple(int(i) for i in data["periodic_dims"])

dynamics_parameters = reference_metadata["dynamics"]["parameters"]
solver_parameters = reference_metadata["solver"]

initial_time = float(solver_parameters["initial_time"])
target_time = float(solver_parameters["target_time"])
solver_accuracy = solver_parameters["accuracy"]

# Verifica della compatibilità delle propagazioni originali
for metric, path in metric_paths.items():
    with np.load(path, allow_pickle=False) as data:
        metadata = json.loads(data["metadata_json"].item())

        same_domain = (
            np.array_equal(data["grid_lo"], grid_lo)
            and np.array_equal(data["grid_hi"], grid_hi)
            and tuple(int(i) for i in data["periodic_dims"]) == periodic_dims
        )

    if not same_domain:
        raise ValueError(
            f"{metric}: dominio o dimensioni periodiche differenti."
        )

    if metadata["dynamics"]["parameters"] != dynamics_parameters:
        raise ValueError(f"{metric}: parametri delle dinamiche differenti.")

    for key in (
        "initial_time",
        "target_time",
        "accuracy",
        "hamiltonian_postprocessor",
    ):
        if metadata["solver"][key] != solver_parameters[key]:
            raise ValueError(
                f"{metric}: impostazione del solver differente: {key}."
            )

if solver_parameters["hamiltonian_postprocessor"] != "backwards_reachable_tube":
    raise ValueError("I file originali non usano il postprocessore BRT atteso.")

# Ricostruzione della griglia con le stesse condizioni al bordo
hj_grid = hj.Grid.from_lattice_parameters_and_boundary_conditions(
    domain=hj.sets.Box(
        lo=jnp.asarray(grid_lo, dtype=jnp.float32),
        hi=jnp.asarray(grid_hi, dtype=jnp.float32),
    ),
    shape=grid_shape,
    periodic_dims=periodic_dims,
)

# Controllo delle coordinate ricostruite
for axis, coordinates in zip(AXES, hj_grid.coordinate_vectors):
    np.testing.assert_allclose(
        np.asarray(coordinates),
        grid[axis],
        rtol=1e-6,
        atol=1e-6,
        err_msg=f"Coordinate ricostruite diverse per {axis}",
    )

dynamics = RelativeVehicle6D(**dynamics_parameters)

solver_settings = hj.SolverSettings.with_accuracy(
    solver_accuracy,
    hamiltonian_postprocessor=hj.solver.backwards_reachable_tube,
)

if V0_mix.shape != tuple(hj_grid.shape):
    raise ValueError("V0_mix non corrisponde alla griglia HJ.")

if not np.all(np.isfinite(V0_mix)):
    raise ValueError("V0_mix contiene NaN o valori infiniti.")

# Stesso tipo numerico usato in compute_brt.py
V0_hj = jnp.asarray(V0_mix, dtype=jnp.float32)

print(f"Metriche: {', '.join(METRICS)}")
print(f"Griglia: {hj_grid.shape}")
print(f"Dimensioni periodiche: {periodic_dims}")
print(f"Accuratezza: {solver_accuracy}")
print(f"Propagazione: {initial_time:g} → {target_time:g} s")
print(f"Dinamiche: {dynamics_parameters}\n")

start = perf_counter()

BRT_mix = hj.step(
    solver_settings,
    dynamics,
    hj_grid,
    initial_time,
    V0_hj,
    target_time,
)

# Attende il completamento effettivo del calcolo JAX
BRT_mix.block_until_ready()

elapsed = perf_counter() - start

if not bool(jnp.all(jnp.isfinite(BRT_mix))):
    raise RuntimeError("Il BRT propagato contiene NaN o valori infiniti.")

print(f"\nPropagazione completata in {elapsed:.2f} s")
print(
    f"Intervallo BRT_mix: "
    f"[{float(jnp.min(BRT_mix)):.4f}, "
    f"{float(jnp.max(BRT_mix)):.4f}]"
)

Metriche: euclidean, dce, ttc, eggert, btn, stn
Griglia: (30, 17, 48, 8, 11, 8)
Dimensioni periodiche: (2,)
Accuratezza: very_high
Propagazione: 0 → -3 s
Dinamiche: {'lf': 1.2, 'lr': 1.5, 'ego_min_acceleration': -7.0, 'ego_max_acceleration': 2.5, 'ego_max_steering_rate': 0.087, 'human_min_acceleration': -7.0, 'human_max_acceleration': 2.5, 'human_max_yaw_rate': 0.15, 'control_mode': 'max', 'disturbance_mode': 'min'}



100%|##########|  3.0000/3.0 [22:33<00:00, 451.24s/sim_s]



Propagazione completata in 1355.66 s
Intervallo BRT_mix: [-5.6883, 1.0000]


In [3]:
from datetime import datetime
from copy import deepcopy

MIX_NAME = "mixed"
output_path = BRT_DIR / f"{MIX_NAME}.npz"

print("Calcolo dei gradienti...")

gradients_mix = hj_grid.grad_values(
    BRT_mix,
    solver_settings.upwind_scheme,
)

gradients_mix.block_until_ready()

V0_save = np.asarray(V0_hj, dtype=np.float32)
BRT_save = np.asarray(BRT_mix, dtype=np.float32)
gradients_save = np.asarray(gradients_mix, dtype=np.float32)

if gradients_save.shape != (*grid_shape, 6):
    raise ValueError("Forma dei gradienti non valida.")

if not np.all(np.isfinite(gradients_save)):
    raise ValueError("I gradienti contengono NaN o valori infiniti.")

# Metadati della nuova combinazione
metadata = deepcopy(reference_metadata)

metadata["created_at"] = datetime.now().astimezone().isoformat()

metadata["metric"] = {
    "name": MIX_NAME,
    "parameters": {
        "metrics": list(METRICS),
        "normalization": "percentile of absolute V0",
        "percentile": float(PERCENTILE),
        "scales": {
            metric: float(normalization_scales[metric])
            for metric in METRICS
        },
        "combination": "pointwise minimum of normalized terminal fields",
    },
    "terminal_value_definition": (
        "V0_mix = min_m(V0_m / percentile(abs(V0_m)))"
    ),
    "sign_convention": "V0 <= 0 unsafe",
}

metadata["arrays"] = {
    "V0_shape": list(V0_save.shape),
    "BRT_shape": list(BRT_save.shape),
    "gradients_shape": list(gradients_save.shape),
    "storage_dtype": "float32",
    "V0_units": "dimensionless",
    "BRT_units": "dimensionless",
}

metadata["software"] = {
    "python_version": sys.version,
    "jax_version": jax.__version__,
}

coordinate_arrays = {
    axis: np.asarray(vector, dtype=np.float32)
    for axis, vector in zip(AXES, hj_grid.coordinate_vectors)
}

output_path.parent.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    output_path,
    V0=V0_save,
    BRT=BRT_save,
    gradients=gradients_save,
    **coordinate_arrays,
    grid_lo=grid_lo,
    grid_hi=grid_hi,
    grid_shape=np.asarray(grid_shape, dtype=np.int32),
    periodic_dims=np.asarray(periodic_dims, dtype=np.int32),
    initial_time=np.float32(initial_time),
    target_time=np.float32(target_time),
    state_names=np.asarray(AXES),
    metric_name=np.asarray(MIX_NAME),
    metadata_json=np.asarray(json.dumps(metadata, indent=4)),
)

print(f"File salvato: {output_path}")
print(f"Dimensione: {output_path.stat().st_size / 1024**2:.1f} MB")

Calcolo dei gradienti...
File salvato: /home/alessio-grisorio/TESI/hj_reachability/results/brt/mixed.npz
Dimensione: 332.3 MB
